# QSP HPC Tools — Feature Walkthrough (C++ backend, offline)

This notebook demonstrates `CppSimulator`, the **primary** simulation backend,
and the content-addressed caching layer that underpins `qsp-hpc-tools`.

**It runs entirely offline** — no HPC cluster, no MATLAB, and no compiled
`qsp_sim` binary. Every simulation call resolves against a small pre-baked
pool committed under `cache/` (built by `build_offline_fixture.py`). The values
are illustrative fixture data; the point is to exercise the real `CppSimulator`
API and its cache-read path without any infrastructure.

Run it from the `examples/` directory.

In [1]:
# Quiet the library's INFO logging so this walkthrough's output stays readable.
import logging
logging.disable(logging.INFO)

## 1. Inputs: priors and calibration targets

Priors define the sampled parameters; calibration targets (MAPLE-style YAMLs) define the observables.

In [2]:
import pandas as pd
import yaml

priors = pd.read_csv("data/priors.csv")
priors

,name,distribution,dist_param1,dist_param2,units,description
0,k_tumor_growth,lognormal,-2.3,0.5,1/day,Tumor growth rate
1,k_tumor_death,lognormal,-3.0,0.4,1/day,Tumor natural death rate
2,k_immune_kill,lognormal,-4.0,0.8,1/day,Immune-mediated kill rate
3,k_immune_recruit,lognormal,-3.5,0.6,1/day,Immune cell recruitment rate
4,k_drug_effect,lognormal,-1.5,0.5,1/day,Drug efficacy multiplier
5,V_tumor_carrying_capacity,lognormal,4.5,0.3,cells,Tumor carrying capacity


In [3]:
# A calibration target: an observable function plus empirical data with units
target = yaml.safe_load(open("data/calibration_targets/control/tumor_volume_day60.yaml"))
print(yaml.dump(target, default_flow_style=False, sort_keys=False))

calibration_target_id: tumor_volume_day60
observable:
  species:
  - V_T.Tumor
  units: cell
  code: "def compute_observable(time, species_dict, constants, ureg):\n    tumor =\
    \ species_dict['V_T.Tumor']\n    idx = np.argmin(np.abs(time.magnitude - 60))\n\
    \    return float(tumor[idx]) * ureg.cell\n"
empirical_data:
  median:
  - 1000000.0
  ci95:
  - - 500000.0
    - 2000000.0
  sample_size: 50



## 2. `CppSimulator` — the primary backend

`CppSimulator` is a callable: requesting *N* simulations returns
`(theta, table)`. It first checks a local pool; here the pool is pre-baked, so
the call is served from cache and the (stub) binary is never executed.

In [4]:
import numpy as np
from qsp_hpc.simulation.cpp_simulator import CppSimulator

def make_sim(scenario, seed=2025, model_version="v1"):
    return CppSimulator(
        priors_csv="data/priors.csv",
        binary_path="fixtures/qsp_sim_stub",       # placeholder; only hashed, never run
        template_xml="fixtures/template_param.xml",
        calibration_targets=f"data/calibration_targets/{scenario}",
        model_version=model_version,
        scenario=scenario,
        cache_dir="cache",
        seed=seed,
        evolve_cache=False,
    )

sim = make_sim("control")
theta, table = sim(200)            # cache hit — no binary, no cluster
print(f"theta:  {theta.shape}   (n_sims x n_params)")
print(f"table:  {table.num_rows} rows, columns = {table.column_names}")

theta:  (200, 6)   (n_sims x n_params)
table:  200 rows, columns = ['sample_index', 'simulation_id', 'status', 'time', 'param:k_tumor_growth', 'param:k_tumor_death', 'param:k_immune_kill', 'param:k_immune_recruit', 'param:k_drug_effect', 'param:V_tumor_carrying_capacity', 'tumor_volume_day60', 'immune_infiltrate_peak', 'time_to_progression']


## 3. Cache hits

Calling again returns the identical cached results — no recomputation.

In [5]:
theta2, _ = sim(200)
assert np.array_equal(theta, theta2), "expected a cache hit"
print("Cache hit: identical parameter matrix returned")

Cache hit: identical parameter matrix returned


## 4. Content-based cache invalidation

The pool directory is keyed by a hash of the *semantic* inputs — priors, the
parameter template, the binary, and the seed. Changing any of them yields a new
hash and therefore a new pool, so stale results are never silently reused.
Cosmetic changes (e.g. a parameter's description) leave the hash untouched.

In [6]:
sim_default = make_sim("control", seed=2025)
sim_reseed  = make_sim("control", seed=999)   # a different semantic input

print(f"seed=2025 -> hash {sim_default.config_hash[:8]}  pool {sim_default.pool_dir.name}")
print(f"seed=999  -> hash {sim_reseed.config_hash[:8]}  pool {sim_reseed.pool_dir.name}")
print("Different hash -> different pool -> no stale cache (the seed=999 pool is empty here).")

seed=2025 -> hash aeb06755  pool v1_aeb06755_control
seed=999  -> hash 7d1a1ef2  pool v1_7d1a1ef2_control
Different hash -> different pool -> no stale cache (the seed=999 pool is empty here).


## 5. Multi-scenario support

The same virtual patients (identical `theta`) are evaluated under independently
cached scenarios. Outputs differ — treatment lowers tumour burden — while the
parameter draws stay aligned, which is what lets downstream inference compare
arms patient-by-patient.

In [7]:
scenarios = ["control", "treatment"]
results = {s: make_sim(s)(200) for s in scenarios}

theta_c, table_c = results["control"]
theta_t, table_t = results["treatment"]
assert np.array_equal(theta_c, theta_t), "theta must be identical across scenarios"
print("Same virtual patients across both scenarios (theta aligned)")

import numpy as np
med = lambda t: float(np.median(t.column("tumor_volume_day60").to_numpy()))
print(f"median tumor_volume_day60 — control: {med(table_c):.1f}, treatment: {med(table_t):.1f}")

Same virtual patients across both scenarios (theta aligned)
median tumor_volume_day60 — control: 96.4, treatment: 70.4


## 6. Posterior-predictive simulation

`simulate_with_parameters(theta)` evaluates the model at user-supplied parameter
draws (typically posterior samples from `qsp-inference`) and returns derived
test statistics, cached under a hash of the input array. Here we reuse the first
50 draws as stand-in posterior samples.

In [8]:
posterior = np.ascontiguousarray(theta_c[:50])
theta_out, ts_table = sim.simulate_with_parameters(posterior)   # cache hit
ts_cols = [c for c in ts_table.column_names if c.startswith("ts:")]
print(f"posterior-predictive: {ts_table.num_rows} samples x {len(ts_cols)} test stats")
print("test-stat columns:", ts_cols)

posterior-predictive: 50 samples x 4 test stats
test-stat columns: ['ts:immune_infiltrate_peak', 'ts:time_to_progression', 'ts:tumor_growth_rate', 'ts:tumor_volume_day60']


## 7. Pool inspection

What's cached on disk, by scenario.

In [9]:
from pathlib import Path

for s in scenarios:
    sim_s = make_sim(s)
    print(f"{s:>9}: {sim_s.get_available_simulations():4d} sims  in  {sim_s.pool_dir.name}")

print("\nPool directories under cache/:")
for p in sorted(Path("cache").glob("v1_*")):
    print(" ", p.name)

  control:  200 sims  in  v1_aeb06755_control
treatment:  200 sims  in  v1_aeb06755_treatment

Pool directories under cache/:
  v1_7d1a1ef2_control
  v1_aeb06755_control
  v1_aeb06755_control_posterior_predictive_19c31ff8
  v1_aeb06755_treatment
